# Fraud Detection: Imbalanced Classification Case Study

Detecting fraudulent transactions in a highly imbalanced dataset where fraud represents only ~2% of all transactions.

## 1. Problem: Credit Card Fraud Detection

Credit card fraud is a classic imbalanced classification problem. Fraudulent transactions are extremely rare (typically 1-2% of all transactions) but cause significant financial damage. The goal is to build a model that flags suspicious transactions in real-time while minimizing disruption to legitimate customers.

**Business constraints:**
- False negatives (missed fraud) = direct financial loss
- False positives (blocked legitimate transactions) = customer friction, support costs

## 2. Generate Synthetic Transaction Data

We simulate realistic transaction data with features typical of fraud detection systems: amount, time of day, location mismatch, merchant type, and user history features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, f1_score,
    precision_score, recall_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

In [ ]:
# Generate synthetic transaction data
# 2% fraud rate with some outliers to mimic real fraud
n_samples = 50000
fraud_ratio = 0.02

X, y = make_classification(
    n_samples=n_samples,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_clusters_per_class=2,
    weights=[1 - fraud_ratio, fraud_ratio],
    flip_y=0.01,
    class_sep=0.8,
    hypercube=True,
    shift=0.0,
    scale=1.0,
    random_state=42
)

# Add some extreme outlier fraud cases
n_outliers = 50
outlier_idx = np.where(y == 1)[0][:n_outliers]
X[outlier_idx] += np.random.normal(0, 5, X[outlier_idx].shape)

# Create meaningful feature names
feature_names = [
    'transaction_amount', 'transaction_hour', 'location_mismatch',
    'merchant_risk_score', 'distance_from_home', 'card_present',
    'avg_transaction_amount_30d', 'num_transactions_24h',
    'num_failed_attempts_7d', 'device_trust_score',
    'ip_country_mismatch', 'shipping_billing_match',
    'new_merchant_flag', 'velocity_1h', 'currency_conversion',
    'login_attempts_24h', 'password_change_days',
    'email_domain_risk', 'phone_verified', 'address_change_days'
]

df = pd.DataFrame(X, columns=feature_names)
df['fraud'] = y

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate: {df["fraud"].mean():.4f} ({df["fraud"].sum()} out of {len(df)})')
df.head()

## 3. Exploratory Data Analysis

Understanding the data distribution and identifying patterns that distinguish fraud from legitimate transactions.

In [ ]:
# Visualization 1: Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4CAF50', '#F44336']
class_counts = df['fraud'].value_counts()
axes[0].bar(['Non-Fraud (0)', 'Fraud (1)'], class_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

axes[1].pie(class_counts.values, labels=['Non-Fraud', 'Fraud'], autopct='%1.2f%%',
            colors=colors, startangle=90, explode=(0, 0.1))
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Feature distributions by class
top_features = ['transaction_amount', 'distance_from_home', 'velocity_1h',
                'num_failed_attempts_7d', 'merchant_risk_score', 'device_trust_score']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for cls, label, color in zip([0, 1], ['Non-Fraud', 'Fraud'], ['#4CAF50', '#F44336']):
        subset = df[df['fraud'] == cls][feat]
        axes[i].hist(subset, bins=50, alpha=0.6, label=label, color=color, density=True)
    axes[i].set_title(f'{feat} by Class', fontsize=12, fontweight='bold')
    axes[i].legend()
    axes[i].set_ylabel('Density')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Challenge: Extreme Class Imbalance

With only 2% fraud, a naive model predicting "no fraud" for every transaction achieves 98% accuracy but is entirely useless. We need specialized techniques:
- Class weighting (penalize mistakes on minority class more)
- Resampling (undersample majority / oversample minority)
- Anomaly detection (treat fraud as outliers)
- Threshold optimization (don't use default 0.5)

In [ ]:
# Split data and scale
X = df.drop('fraud', axis=1)
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} samples, fraud rate: {y_train.mean():.4f}')
print(f'Test set: {X_test.shape[0]} samples, fraud rate: {y_test.mean():.4f}')

## 5. Methods

We evaluate multiple approaches to handle the imbalance.

In [ ]:
# 5a. Baseline: predict all non-fraud
y_pred_baseline = np.zeros_like(y_test)

print('=== Baseline: Predict All Non-Fraud ===')
print(f'Precision: {precision_score(y_test, y_pred_baseline):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_baseline):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_baseline):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_pred_baseline):.4f}')

In [ ]:
# 5b. Logistic Regression with class weights
lr_weighted = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_weighted.fit(X_train_scaled, y_train)
y_pred_lr = lr_weighted.predict(X_test_scaled)
y_prob_lr = lr_weighted.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression (Balanced Weights) ===')
print(f'Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_lr):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_prob_lr):.4f}')

In [ ]:
# 5c. Random Forest with balanced subsampling
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced_subsample',
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_balanced.fit(X_train_scaled, y_train)
y_pred_rf = rf_balanced.predict(X_test_scaled)
y_prob_rf = rf_balanced.predict_proba(X_test_scaled)[:, 1]

print('=== Random Forest (Balanced Subsampling) ===')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_rf):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_prob_rf):.4f}')

In [ ]:
# 5d. Isolation Forest (unsupervised anomaly detection)
iso_forest = IsolationForest(
    contamination=0.02,
    random_state=42,
    n_estimators=200
)
iso_forest.fit(X_train_scaled)

# Isolation Forest returns -1 for outliers (fraud) and 1 for inliers
y_pred_if = iso_forest.predict(X_test_scaled)
y_pred_if = np.where(y_pred_if == -1, 1, 0)

# Anomaly scores (lower = more anomalous)
y_score_if = -iso_forest.score_samples(X_test_scaled)

print('=== Isolation Forest (Unsupervised) ===')
print(f'Precision: {precision_score(y_test, y_pred_if):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_if):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_if):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_score_if):.4f}')

In [ ]:
# 5e. SMOTE oversampling + Random Forest
smote_pipeline = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.2, random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ))
])
smote_pipeline.fit(X_train_scaled, y_train)
y_pred_smote = smote_pipeline.predict(X_test_scaled)
y_prob_smote = smote_pipeline.predict_proba(X_test_scaled)[:, 1]

print('=== SMOTE + Random Forest ===')
print(f'Precision: {precision_score(y_test, y_pred_smote):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_smote):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_smote):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_prob_smote):.4f}')

## 6. Model Comparison

Compare all models using proper imbalanced classification metrics.

In [ ]:
# Visualization 4: Model comparison
models = ['Baseline', 'LogReg
(weighted)', 'Random
Forest', 'Isolation
Forest', 'SMOTE+RF']
precisions = [
    precision_score(y_test, y_pred_baseline),
    precision_score(y_test, y_pred_lr),
    precision_score(y_test, y_pred_rf),
    precision_score(y_test, y_pred_if),
    precision_score(y_test, y_pred_smote)
]
recalls = [
    recall_score(y_test, y_pred_baseline),
    recall_score(y_test, y_pred_lr),
    recall_score(y_test, y_pred_rf),
    recall_score(y_test, y_pred_if),
    recall_score(y_test, y_pred_smote)
]
f1_scores = [
    f1_score(y_test, y_pred_baseline),
    f1_score(y_test, y_pred_lr),
    f1_score(y_test, y_pred_rf),
    f1_score(y_test, y_pred_if),
    f1_score(y_test, y_pred_smote)
]
pr_aucs = [
    average_precision_score(y_test, y_pred_baseline),
    average_precision_score(y_test, y_prob_lr),
    average_precision_score(y_test, y_prob_rf),
    average_precision_score(y_test, y_score_if),
    average_precision_score(y_test, y_prob_smote)
]

x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - 1.5*width, precisions, width, label='Precision', color='#2196F3')
bars2 = ax.bar(x - 0.5*width, recalls, width, label='Recall', color='#FF9800')
bars3 = ax.bar(x + 0.5*width, f1_scores, width, label='F1 Score', color='#4CAF50')
bars4 = ax.bar(x + 1.5*width, pr_aucs, width, label='PR-AUC', color='#9C27B0')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Imbalanced Classification Metrics', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=11)
ax.axhline(y=0.02, color='gray', linestyle='--', alpha=0.5, label='Random (PR-AUC baseline)')

for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 7. Threshold Optimization on PR Curve

Default threshold of 0.5 is rarely optimal for imbalanced data. We optimize on the precision-recall curve.

In [ ]:
# Use the best model (Random Forest) for threshold optimization
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_rf)
f1_scores_thresh = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

# Visualization 5: PR Curve with optimal threshold
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PR Curve
axes[0].plot(recalls, precisions, 'b-', linewidth=2, label=f'RF (PR-AUC = {average_precision_score(y_test, y_prob_rf):.3f})')
axes[0].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.7, label=f'Random ({y_test.mean():.3f})')
optimal_idx = np.argmax(f1_scores_thresh)
optimal_thresh = thresholds[optimal_idx]
axes[0].scatter(recalls[optimal_idx], precisions[optimal_idx], color='red', s=150,
                zorder=5, label=f'Optimal (thresh={optimal_thresh:.3f})')
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# Threshold vs Metrics
axes[1].plot(thresholds, precisions[:-1], 'b-', label='Precision', linewidth=2)
axes[1].plot(thresholds, recalls[:-1], 'orange', label='Recall', linewidth=2)
axes[1].plot(thresholds, f1_scores_thresh, 'g-', label='F1 Score', linewidth=2)
axes[1].axvline(x=optimal_thresh, color='red', linestyle='--', alpha=0.7)
axes[1].scatter(optimal_thresh, f1_scores_thresh[optimal_idx], color='red', s=150, zorder=5)
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Threshold vs Metrics', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Optimal threshold (max F1): {optimal_thresh:.4f}')
print(f'  => Precision: {precisions[optimal_idx]:.4f}')
print(f'  => Recall: {recalls[optimal_idx]:.4f}')
print(f'  => F1 Score: {f1_scores_thresh[optimal_idx]:.4f}')

## 8. Cost-Benefit Analysis

Not all errors are equal. Let's assign costs and find the threshold that minimizes total business cost.

**Cost assumptions:**
- False negative (missed fraud): $200 (chargeback + investigation + lost merchandise)
- False positive (blocked legitimate): $25 (customer service call + lost fee + goodwill)

In [ ]:
# Cost-benefit analysis
cost_fn = 200   # cost of missed fraud
cost_fp = 25    # cost of false alarm (blocked legitimate tx)

total_costs = []
fp_rates = []
fn_rates = []

for thresh in thresholds:
    y_pred_thresh = (y_prob_rf >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred_thresh)
    tn, fp, fn, tp = cm.ravel()
    total_cost = fn * cost_fn + fp * cost_fp
    total_costs.append(total_cost)
    fp_rates.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    fn_rates.append(fn / (fn + tp) if (fn + tp) > 0 else 0)

min_cost_idx = np.argmin(total_costs)
optimal_cost_thresh = thresholds[min_cost_idx]

# Visualization 7: Cost curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total cost vs threshold
axes[0].plot(thresholds, total_costs, 'purple', linewidth=2)
axes[0].axvline(x=optimal_cost_thresh, color='red', linestyle='--', alpha=0.7)
axes[0].scatter(optimal_cost_thresh, total_costs[min_cost_idx], color='red', s=150, zorder=5)
axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Total Cost ($)', fontsize=12)
axes[0].set_title('Cost vs Threshold', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# FP and FN rates vs threshold
axes[1].plot(thresholds, fp_rates, 'b-', label='False Positive Rate', linewidth=2)
axes[1].plot(thresholds, fn_rates, 'orange', label='False Negative Rate', linewidth=2)
axes[1].axvline(x=optimal_cost_thresh, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('Error Rate', fontsize=12)
axes[1].set_title('FP / FN Rates vs Threshold', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'=== Cost-Benefit Analysis Results ===')
print(f'Cost optimal threshold: {optimal_cost_thresh:.4f}')
print(f'Minimum total cost: ${total_costs[min_cost_idx]:,.0f}')

# Apply optimal threshold
y_pred_optimal = (y_prob_rf >= optimal_cost_thresh).astype(int)
print(f'\nAt optimal threshold:')
print(f'  Precision: {precision_score(y_test, y_pred_optimal):.4f}')
print(f'  Recall: {recall_score(y_test, y_pred_optimal):.4f}')
print(f'  F1 Score: {f1_score(y_test, y_pred_optimal):.4f}')

## 9. Confusion Matrix

Visualizing the final model's performance with the cost-optimized threshold.

In [ ]:
# Visualization 8: Confusion matrix
cm = confusion_matrix(y_test, y_pred_optimal)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Non-Fraud', 'Fraud'],
            yticklabels=['Non-Fraud', 'Fraud'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Cost-Optimized Threshold)', fontsize=14, fontweight='bold')

tn, fp, fn, tp = cm.ravel()
total_cost = fn * cost_fn + fp * cost_fp
plt.figtext(0.5, -0.05,
    f'Total Business Cost: ${total_cost:,}  |  ' +
    f'FN (Missed Fraud): {fn} × ${cost_fn} = ${fn * cost_fn:,}  |  ' +
    f'FP (Blocked Legitimate): {fp} × ${cost_fp} = ${fp * cost_fp:,}',
    ha='center', fontsize=11, bbox=dict(facecolor='lightyellow', alpha=0.8, boxstyle='round'))

plt.tight_layout()
plt.show()

## 10. Feature Importance

Understanding which features are most predictive of fraud helps in model interpretation and deployment decisions.

In [ ]:
# Visualization 9: Feature importance
importances = rf_balanced.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(indices)))
bars = ax.barh(range(len(indices)), importances[indices], color=colors)
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names[i] for i in indices], fontsize=10)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, importances[indices]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Summary report
print('='*70)
print('FRAUD DETECTION: FINAL SUMMARY')
print('='*70)
print(f'\nBest Model: Random Forest with cost-optimized threshold ({optimal_cost_thresh:.3f})')
print(f'\n=== Performance at Optimal Threshold ===')
print(f'{'Precision:':25s} {precision_score(y_test, y_pred_optimal):.4f}')
print(f'{'Recall:':25s} {recall_score(y_test, y_pred_optimal):.4f}')
print(f'{'F1 Score:':25s} {f1_score(y_test, y_pred_optimal):.4f}')
print(f'{'PR-AUC:':25s} {average_precision_score(y_test, y_prob_rf):.4f}')
print(f'\n=== Business Impact (${cost_fn}/FN, ${cost_fp}/FP) ===')
print(f'{'False Negatives (missed fraud):':35s} {fn}')
print(f'{'False Positives (blocked legitimate):':35s} {fp}')
print(f'{'Total Business Cost:':35s} ${total_cost:,}')
print(f'\n=== Key Fraud Signals (Top 5) ===')
for i in range(5):
    idx = indices[i]
    print(f'  {i+1}. {feature_names[idx]:30s} (importance: {importances[idx]:.4f})')
print('\n' + '='*70)